# Normalizacao do Relatorio da BRACOFER (Contas a Pagar)

Este notebook le o arquivo bruto gerado pelo sistema **Supply** (relatorio de Contas a Pagar da Bracofer), permitindo selecionar o arquivo do mes desejado na pasta `02-Referencias/Supply_bracofer/`.

A execucao gera dois arquivos de saida:

1. **Relatorio Contas a Pagar {periodo} - Normalizado.xlsx** — versao tabular do relatorio em blocos do Supply (cada pagamento em uma linha, com CC e Classificacao Financeira como colunas).
2. **BRACOFER_fechamento_{periodo}.xlsx** — layout `FECHAMENTO_ODBC`, com os Centros de Custo e Planos de Contas convertidos para o padrao **SAGI**:
   - CCs: ramo `1.3.1.x` (Bracofer / Presidente Prudente — Administrativo, Comercial, Operacional, Logistica, Corte e Dobra)
   - PCs: mapa Supply → SAGI, roteamento contextual do codigo **2010101** (Servicos), e descricoes alinhadas ao CSV `sagi_rel_plano_conta.csv` (celula *Plano de Contas — Mapeamento Supply → SAGI*)

In [24]:
from pathlib import Path
from datetime import datetime
import re
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

# ── Parametros ──────────────────────────────────────────────────────────────
MES_REFERENCIA = None      # Ex.: "05/2026". Se None, tenta inferir do nome do arquivo.
SUPPLY_SUBPASTA = "Maio"   # Ex.: "Maio", "Abril", "Março". None = raiz ou busca recursiva.
ARQUIVO_ENTRADA_NOME = "Relatório Contas a Pagar de 01-05-2026 até 31-05-2026.xlsx"
# ────────────────────────────────────────────────────────────────────────────

REFS_DIR = Path("../../02-Referencias")
SUPPLY_DIR = REFS_DIR / "Supply_bracofer"

if not SUPPLY_DIR.exists():
    raise FileNotFoundError(f"Pasta nao encontrada: {SUPPLY_DIR.resolve()}")

def _iter_supply_xlsx():
    """Arquivos .xlsx de Contas a Pagar (raiz, subpasta do mes ou recursivo)."""
    bases = []
    if SUPPLY_SUBPASTA:
        sub = SUPPLY_DIR / SUPPLY_SUBPASTA
        if sub.is_dir():
            bases.append(sub)
    bases.append(SUPPLY_DIR)
    vistos: set[Path] = set()
    for base in bases:
        for p in base.rglob("*.xlsx"):
            if p in vistos or p.name.startswith("~$"):
                continue
            if "contas a pagar" not in p.name.lower():
                continue
            if "normalizado" in p.name.lower():
                continue
            vistos.add(p)
            yield p

if ARQUIVO_ENTRADA_NOME:
    candidatos = []
    if SUPPLY_SUBPASTA:
        candidatos.append(SUPPLY_DIR / SUPPLY_SUBPASTA / ARQUIVO_ENTRADA_NOME)
    candidatos.append(SUPPLY_DIR / ARQUIVO_ENTRADA_NOME)
    for p in SUPPLY_DIR.rglob(ARQUIVO_ENTRADA_NOME):
        if p not in candidatos:
            candidatos.append(p)
    candidatos = [p for p in candidatos if p.exists()]
else:
    candidatos = sorted(_iter_supply_xlsx(), key=lambda p: p.stat().st_mtime)

if not candidatos:
    raise FileNotFoundError(
        f"Nenhum arquivo de Contas a Pagar encontrado em {SUPPLY_DIR.resolve()}"
    )

ARQUIVO_ENTRADA = candidatos[-1]
if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo de entrada nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

# Saidas na mesma pasta do arquivo bruto (ex.: Supply_bracofer/Maio/)
SUPPLY_MES_DIR = ARQUIVO_ENTRADA.parent

def _extrair_mes_ano(nome_arquivo: str) -> tuple[str | None, str | None]:
    m = re.search(r"de\s*\d{2}-(\d{2})-(\d{4})\s*(?:ate|até)\s*\d{2}-\d{2}-\d{4}", nome_arquivo, flags=re.IGNORECASE)
    if m:
        return m.group(1), m.group(2)
    m2 = re.search(r"(\d{2})[-_](\d{4})", nome_arquivo)
    if m2:
        return m2.group(1), m2.group(2)
    return None, None

if MES_REFERENCIA:
    mes_num, ano = MES_REFERENCIA.split("/")
else:
    mes_num, ano = _extrair_mes_ano(ARQUIVO_ENTRADA.name)
    if not mes_num or not ano:
        dt_ref = datetime.fromtimestamp(ARQUIVO_ENTRADA.stat().st_mtime)
        mes_num = dt_ref.strftime("%m")
        ano = dt_ref.strftime("%Y")

SUFIXO_PERIODO = f"{mes_num}-{ano}"
ARQUIVO_SAIDA = SUPPLY_MES_DIR / f"Relatorio Contas a Pagar {SUFIXO_PERIODO} - Normalizado.xlsx"

print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Saida:   {ARQUIVO_SAIDA.resolve()}")
print(f"Mes referencia: {mes_num}/{ano}")

Entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Supply_bracofer\Maio\Relatório Contas a Pagar de 01-05-2026 até 31-05-2026.xlsx
Saida:   C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Supply_bracofer\Maio\Relatorio Contas a Pagar 05-2026 - Normalizado.xlsx
Mes referencia: 05/2026


In [25]:
def _norm_text(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

def _to_number(v):
    if pd.isna(v):
        return pd.NA
    if isinstance(v, (int, float)):
        return float(v)

    s = str(v).strip()
    if not s:
        return pd.NA

    # Se vier no padrao brasileiro: 1.234,56
    if "," in s:
        s = s.replace(".", "").replace(",", ".")

    try:
        return float(s)
    except ValueError:
        return pd.NA

def _montar_observacao(c2) -> str:
    """
    Observacao com referencia a NF: apenas 'REFERENTE A NF' + 9 digitos.
    O que vem na coluna 4 antes de ' - ' costuma ser CNPJ/codigo do credor (ja na coluna credor),
    nao parte do numero da NF — nao concatenar com a coluna 4.
    """
    import re

    c2s = _norm_text(c2)
    if not c2s:
        return c2s
    # Case-insensitive (o Excel pode trazer "Referente a NF" em minusculas).
    m = re.match(r"(?is)referente\s+[aàá]\s+nf", c2s)
    if not m:
        return c2s
    resto = c2s[m.end() :].strip()
    digitos = "".join(c for c in resto if c.isdigit())
    if not digitos:
        return c2s
    if len(digitos) >= 9:
        nove = digitos[:9]
    else:
        nove = digitos.zfill(9)
    return f"REFERENTE A NF {nove}"

def parse_relatorio_contas_pagar(path_xlsx: Path) -> pd.DataFrame:
    import re

    raw = pd.read_excel(path_xlsx, sheet_name=0, header=None, dtype=object)
    layout = "wide" if raw.shape[1] >= 22 else "compact"

    def _detectar_variante_wide(df_raw: pd.DataFrame) -> str:
        """wide_cf = fornecedor na col 4 + CF em linhas; wide_inline = fornecedor na col 3."""
        for _, hdr in df_raw.head(40).iterrows():
            c1 = _norm_text(hdr[1]).lower()
            if "numero do documento" not in c1 and "número do documento" not in c1:
                continue
            c3 = _norm_text(hdr[3]).lower()
            c4 = _norm_text(hdr[4]).lower()
            if c3 == "fornecedor":
                return "wide_inline"
            if c4 == "fornecedor":
                return "wide_cf"
        return "wide_cf"

    variante_wide = _detectar_variante_wide(raw) if layout == "wide" else ""
    if layout == "wide" and variante_wide == "wide_cf" and raw.shape[1] >= 28:
        variante_wide = "wide_cf_v2"

    _COLS_WIDE_CF = {
        "wide_cf": {
            "emissao": 6, "vencimento": 7, "pagamento": 9, "valor": 12,
            "desconto": 17, "juros": 18, "multa": 20, "devolucao": 21,
            "liquido": 22, "empresa": 23, "cf_valor": 8, "cf_cc": 16, "cf_cc_valor": 22,
        },
        "wide_cf_v2": {
            "emissao": 8, "vencimento": 10, "pagamento": 12, "valor": 15,
            "desconto": 21, "juros": 23, "multa": 25, "devolucao": 26,
            "liquido": 27, "empresa": 28, "cf_valor": 11, "cf_cc": 20, "cf_cc_valor": 27,
        },
    }

    _SKIP_C1 = {
        "Numero do Documento",
        "Número do Documento",
        "Classificacao Financeira",
        "Classificação Financeira",
        "Codigo",
        "Código",
    }

    def _eh_linha_rodape(*vals) -> bool:
        txt = " ".join(_norm_text(v).lower() for v in vals)
        if "desenvolvido por softland" in txt:
            return True
        if re.search(r"p[áa]gina\s+\d+\s+de\s+\d+", txt):
            return True
        return False

    def _extrair_cc_secao(texto: str) -> str:
        m = re.search(r"centro\s+de\s+custo\s*:\s*\d+\s*-\s*(.+)", texto, flags=re.IGNORECASE)
        return m.group(1).strip() if m else ""

    def _eh_linha_documento(c1: str, c2: str, c3: str, c4: str) -> bool:
        if not c1 or c1 in _SKIP_C1:
            return False
        c2_low = c2.lower()
        if c2_low.startswith("observa") or c2_low.startswith("centro de custo"):
            return False
        if layout == "wide":
            if variante_wide == "wide_inline":
                return bool(c3 and " - " in c3)
            return bool(c4)
        return bool(c3 and " - " in c3)

    def _nova_linha_documento(row, fornecedor: str, cc_secao: str) -> dict:
        if layout == "wide" and variante_wide == "wide_inline":
            return {
                "numero_documento": _norm_text(row[1]),
                "descricao_documento": _norm_text(row[2]),
                "observacao": _montar_observacao(row[2]),
                "fornecedor": fornecedor,
                "emissao": _norm_text(row[7]),
                "vencimento": _norm_text(row[9]),
                "pagamento": _norm_text(row[10]),
                "valor_documento": _to_number(row[13]),
                "desconto": _to_number(row[17]),
                "juros": _to_number(row[19]),
                "multa": _to_number(row[21]),
                "devolucao": _to_number(row[22]),
                "liquido": _to_number(row[23]),
                "empresa": _norm_text(row[24]),
                "classificacao_financeira_codigo": "",
                "classificacao_financeira_descricao": "",
                "classificacao_financeira_valor": pd.NA,
                "centro_custos_descricao": cc_secao,
                "centro_custos_valor": _to_number(row[23]),
            }
        if layout == "wide" and variante_wide in _COLS_WIDE_CF:
            c = _COLS_WIDE_CF[variante_wide]
            return {
                "numero_documento": _norm_text(row[1]),
                "descricao_documento": _norm_text(row[2]),
                "observacao": _montar_observacao(row[2]),
                "fornecedor": fornecedor,
                "emissao": _norm_text(row[c["emissao"]]),
                "vencimento": _norm_text(row[c["vencimento"]]),
                "pagamento": _norm_text(row[c["pagamento"]]),
                "valor_documento": _to_number(row[c["valor"]]),
                "desconto": _to_number(row[c["desconto"]]),
                "juros": _to_number(row[c["juros"]]),
                "multa": _to_number(row[c["multa"]]),
                "devolucao": _to_number(row[c["devolucao"]]),
                "liquido": _to_number(row[c["liquido"]]),
                "empresa": _norm_text(row[c["empresa"]]),
                "classificacao_financeira_codigo": "",
                "classificacao_financeira_descricao": "",
                "classificacao_financeira_valor": pd.NA,
                "centro_custos_descricao": "",
                "centro_custos_valor": pd.NA,
            }
        return {
            "numero_documento": _norm_text(row[1]),
            "descricao_documento": _norm_text(row[2]),
            "observacao": _montar_observacao(row[2]),
            "fornecedor": fornecedor,
            "emissao": _norm_text(row[5]),
            "vencimento": _norm_text(row[6]),
            "pagamento": _norm_text(row[7]),
            "valor_documento": _to_number(row[10]),
            "desconto": _to_number(row[13]),
            "juros": _to_number(row[14]),
            "multa": _to_number(row[16]),
            "devolucao": _to_number(row[17]),
            "liquido": _to_number(row[18]),
            "empresa": _norm_text(row[19]),
            "classificacao_financeira_codigo": "",
            "classificacao_financeira_descricao": "",
            "classificacao_financeira_valor": pd.NA,
            "centro_custos_descricao": cc_secao,
            "centro_custos_valor": _to_number(row[10]),
        }

    registros = []
    atual = None
    cc_secao_atual = ""

    for _, row in raw.iterrows():
        c1 = _norm_text(row[1])
        c2 = _norm_text(row[2])
        c3 = _norm_text(row[3])
        c4 = _norm_text(row[4])

        if _eh_linha_rodape(c1, c2, c3, c4):
            continue

        cc_linha = _extrair_cc_secao(c2) or _extrair_cc_secao(c1)
        if cc_linha:
            cc_secao_atual = cc_linha
            continue

        obs_txt = ""
        if c2.lower().startswith("observa"):
            obs_txt = c2
        elif c1.lower().startswith("observa"):
            obs_txt = c1
        if obs_txt and atual is not None:
            obs_extra = obs_txt.split(":", 1)[-1].strip() if ":" in obs_txt else obs_txt
            if obs_extra:
                base = _norm_text(atual.get("observacao"))
                atual["observacao"] = f"{base} | {obs_extra}".strip(" |")
            continue

        if _eh_linha_documento(c1, c2, c3, c4):
            if layout == "wide":
                fornecedor = c3 if variante_wide == "wide_inline" else c4
            else:
                fornecedor = c3
            atual = _nova_linha_documento(row, fornecedor, cc_secao_atual)
            registros.append(atual)
            continue

        if atual is None or layout != "wide" or variante_wide not in _COLS_WIDE_CF:
            continue

        c_cf = _COLS_WIDE_CF[variante_wide]
        cf_valor = row[c_cf["cf_valor"]]
        if c2 and c3 and (cf_valor is not None or c1 == "") and c2 not in _SKIP_C1:
            if c2.replace(" ", "").isalnum() and c3:
                atual["classificacao_financeira_codigo"] = c2
                atual["classificacao_financeira_descricao"] = c3
                atual["classificacao_financeira_valor"] = _to_number(cf_valor)
                atual["centro_custos_descricao"] = _norm_text(row[c_cf["cf_cc"]]) or cc_secao_atual
                atual["centro_custos_valor"] = _to_number(row[c_cf["cf_cc_valor"]])

    df = pd.DataFrame(registros)
    globals()["_LAYOUT_SUPPLY_DETECTADO"] = (
        f"{layout}/{variante_wide}" if layout == "wide" and variante_wide else layout
    )

    # Remove linhas totalmente vazias (se houver algum ruido).
    if not df.empty:
        chave = ["numero_documento", "fornecedor"]
        df = df[df[chave].apply(lambda s: any(_norm_text(x) for x in s), axis=1)].copy()

        # Barreira final para evitar rodape no dataset normalizado.
        # Nao usar "softland sistemas" sozinho — e razao social de fornecedor valida.
        padrao_rodape = r"desenvolvido\s+por\s+softland"
        mascara_rodape = (
            df["numero_documento"].astype(str).str.contains(padrao_rodape, case=False, regex=True, na=False)
            | df["descricao_documento"].astype(str).str.contains(padrao_rodape, case=False, regex=True, na=False)
            | df["fornecedor"].astype(str).str.contains(padrao_rodape, case=False, regex=True, na=False)
            | df["observacao"].astype(str).str.contains(padrao_rodape, case=False, regex=True, na=False)
        )
        df = df.loc[~mascara_rodape].copy()

    return df

df_norm = parse_relatorio_contas_pagar(ARQUIVO_ENTRADA)
_layout = globals().get("_LAYOUT_SUPPLY_DETECTADO", "?")
_sem_pc = int((df_norm["classificacao_financeira_codigo"].astype(str).str.strip() == "").sum()) if len(df_norm) else 0
print(f"Layout detectado: {_layout} | Linhas: {len(df_norm)}")
if _sem_pc and len(df_norm):
    print(
        f"[AVISO] {_sem_pc} linha(s) sem Classificacao Financeira. "
        "Reexporte do Supply com 'Exibir Classificacao Financeira' marcado "
        "(ver nota [[Gerando Relatório de Contas a Pagar da Bracofer (Supply)]])."
    )
df_norm.head(10)

c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Layout detectado: wide/wide_cf_v2 | Linhas: 266


,numero_documento,descricao_documento,observacao,fornecedor,emissao,vencimento,pagamento,valor_documento,desconto,juros,multa,devolucao,liquido,empresa,classificacao_financeira_codigo,classificacao_financeira_descricao,classificacao_financeira_valor,centro_custos_descricao,centro_custos_valor
0,000000000-001,REFERENTE A NF 000000000,REFERENTE A NF 000000000 | [GIULIANA] EMBRATEL...,02667694000140 - TELMEX DO BRASIL S/A,22/05/26,25/05/26,25/05/26,58.69,0.0,0.0,0.0,0.0,58.69,003,02010101,Serviços,58.69,Administrativo,58.69
1,000000003-001,REFERENTE A NF 000000003,REFERENTE A NF 000000003 | PEDIDO(S) ASSOCIADO...,46007927000154 - ALEX FERREIRA ARCE 33602634809,04/05/26,15/05/26,15/05/26,391.96,0.0,0.0,0.0,0.0,391.96,003,02010101,Serviços,391.96,Administrativo,391.96
2,000000005-001,REFERENTE A NF 000000005,REFERENTE A NF 000000005 | PEDIDO(S) ASSOCIADO...,66235712000106 - INDUSTRIA DE ROLDANAS SEMENSI...,18/05/26,25/05/26,25/05/26,1500.00,0.0,0.0,0.0,0.0,1500.00,003,0301,Mercadoria p/Revenda,1500.00,Comercial,1500.00
3,000000039-001,REFERENTE A NF 000000039,REFERENTE A NF 000000039 | PEDIDO(S) ASSOCIADO...,57360768000193 - PATRICIA ALVES TOLEDO,05/05/26,15/05/26,15/05/26,700.00,0.0,0.0,0.0,0.0,700.00,003,02010101,Serviços,700.00,Administrativo,700.00
4,000000042-001,REFERENTE A NF 000000042,REFERENTE A NF 000000042 | PEDIDO(S) ASSOCIADO...,55599387000136 - SANTANA E DA SILVA CONSULTORI...,05/05/26,15/05/26,15/05/26,4000.00,0.0,0.0,0.0,0.0,4000.00,003,02010101,Serviços,4000.00,Operacional,4000.00
5,000000107-001,REFERENTE A NF 000000107,REFERENTE A NF 000000107 | PEDIDO(S) ASSOCIADO...,52596265000106 - GALAS TRAFAGO PAGO LTDA,05/05/26,15/05/26,15/05/26,1000.00,0.0,0.0,0.0,0.0,1000.00,003,02010101,Serviços,1000.00,Administrativo,1000.00
6,000000119-001,REFERENTE A NF 000000119,REFERENTE A NF 000000119 | PEDIDO(S) ASSOCIADO...,30905135000167 - MARAIZA FEITOSA MARTINS PARDI...,12/05/26,15/05/26,15/05/26,1000.00,0.0,0.0,0.0,0.0,1000.00,003,02040115,Propaganda e Marketing,1000.00,Administrativo,1000.00
7,000000196-001,REFERENTE A NF 000000196,REFERENTE A NF 000000196 | PEDIDO(S) ASSOCIADO...,00395519000116 - FUNDACAO DE CIENCIA TECNOLOGI...,05/05/26,15/05/26,18/05/26,11880.00,0.0,0.0,0.0,0.0,11880.00,003,02010101,Serviços,11880.00,Administrativo,11880.00
8,000000257-001,REFERENTE A NF 000000257,REFERENTE A NF 000000257 | PEDIDO(S) ASSOCIADO...,43104416000162 - BRUNO GRIGOLETTO PEREIRA 4373...,15/05/26,25/05/26,25/05/26,250.00,0.0,0.0,0.0,0.0,250.00,003,02010101,Serviços,250.00,Maq. e Equipamentos,250.00
9,000000310-001,REFERENTE A NF 000000310,REFERENTE A NF 000000310 | PEDIDO(S) ASSOCIADO...,25533997000176 - OSSUCCI CONTABILIDADE LTDA,28/05/26,15/06/26,-,2120.00,0.0,0.0,0.0,0.0,2120.00,003,02010101,Serviços,2120.00,Administrativo,2120.00


In [26]:
print(f"Total de linhas normalizadas: {len(df_norm)}")
print("\nColunas:")
print(df_norm.columns.tolist())

df_norm.sample(min(5, len(df_norm)), random_state=42)

Total de linhas normalizadas: 266

Colunas:
['numero_documento', 'descricao_documento', 'observacao', 'fornecedor', 'emissao', 'vencimento', 'pagamento', 'valor_documento', 'desconto', 'juros', 'multa', 'devolucao', 'liquido', 'empresa', 'classificacao_financeira_codigo', 'classificacao_financeira_descricao', 'classificacao_financeira_valor', 'centro_custos_descricao', 'centro_custos_valor']


,numero_documento,descricao_documento,observacao,fornecedor,emissao,vencimento,pagamento,valor_documento,desconto,juros,multa,devolucao,liquido,empresa,classificacao_financeira_codigo,classificacao_financeira_descricao,classificacao_financeira_valor,centro_custos_descricao,centro_custos_valor
181,D260522-003,,,5419 - COOPERATIVA SICREDI,14/05/26,14/05/26,14/05/26,292.84,0.0,0.0,0.0,0.0,292.84,003,02040208,Taxa Adm Cartão,292.84,Administrativo,292.84
119,3000320096002,REFERENTE A NF 000320096,REFERENTE A NF 000320096 | PEDIDO(S) ASSOCIADO...,05477207000175 - AMAZON ACO INDUSTRIA E COMERC...,01/05/26,05/06/26,05/06/26,30303.59,0.0,0.0,0.0,0.0,30303.59,003,0301,Mercadoria p/Revenda,30303.59,Comercial,30303.59
139,9017040382002,REFERENTE A NF 001014842,REFERENTE A NF 001014842 | PEDIDO(S) ASSOCIADO...,07358761004580 - GERDAU ACOS LONGOS S.A. - SAO...,21/05/26,11/06/26,-,2429.60,0.0,0.0,0.0,0.0,2429.60,003,0301,Mercadoria p/Revenda,2429.60,Comercial,2429.60
216,D260608-002,LOCAÇÃO DE VEICULO,LOCAÇÃO DE VEICULO | REFERENTE A LOCAÇÃO DE VE...,06916551000186 - RSE COMERCIO DE FERRO ACO E L...,27/05/26,25/06/26,-,8500.00,0.0,0.0,0.0,0.0,8500.00,003,02040101,Aluguel,8500.00,Maq. e Equipamentos,8500.00
45,000028472-001,REFERENTE A NF 000028472,REFERENTE A NF 000028472 | CRISTINA - REF EXAM...,40141050000103 - POLIVIDA CLINICA MEDICA LTDA,07/05/26,15/05/26,15/05/26,156.00,0.0,0.0,0.0,0.0,156.00,003,02010101,Serviços,156.00,Administrativo,156.00


In [27]:
from openpyxl.styles import Font

# Se o arquivo principal estiver aberto no Excel, salva uma copia alternativa.
arquivo_saida_exec = ARQUIVO_SAIDA
try:
    with pd.ExcelWriter(arquivo_saida_exec, engine="openpyxl") as writer:
        sheet_name = "contas_pagar_normalizado"
        df_norm.to_excel(writer, sheet_name=sheet_name, index=False)

        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font
except PermissionError:
    arquivo_saida_exec = ARQUIVO_SAIDA.with_name(
        ARQUIVO_SAIDA.stem + "_novo" + ARQUIVO_SAIDA.suffix
    )
    with pd.ExcelWriter(arquivo_saida_exec, engine="openpyxl") as writer:
        sheet_name = "contas_pagar_normalizado"
        df_norm.to_excel(writer, sheet_name=sheet_name, index=False)

        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font

print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")

Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Supply_bracofer\Maio\Relatorio Contas a Pagar 05-2026 - Normalizado.xlsx


## Segunda normalizacao para padrao de fechamento

Nesta etapa, usamos como fonte principal o arquivo `Relatorio Contas a Pagar {periodo} - Normalizado.xlsx` (saida da etapa anterior) e convertemos para o layout do `FECHAMENTO_ODBC_{AAAA}_{MM}.xlsx`.

Se o modelo do mes nao existir, o notebook usa automaticamente o ultimo `FECHAMENTO_ODBC_*.xlsx` disponivel em `02-Referencias/`.

In [28]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

from pathlib import Path
import pandas as pd

REFS_DIR = Path("../../02-Referencias")
SUPPLY_DIR = REFS_DIR / "Supply_bracofer"
SUPPLY_MES_DIR = globals().get("SUPPLY_MES_DIR") or SUPPLY_DIR

ARQUIVO_NORMALIZADO_PRINCIPAL = globals().get("ARQUIVO_SAIDA")
if not ARQUIVO_NORMALIZADO_PRINCIPAL:
    normalizados = sorted(
        [
            p
            for p in SUPPLY_DIR.rglob("Relatorio Contas a Pagar * - Normalizado.xlsx")
            if not p.name.startswith("~$")
        ],
        key=lambda p: p.stat().st_mtime,
    )
    ARQUIVO_NORMALIZADO_PRINCIPAL = (
        normalizados[-1] if normalizados else SUPPLY_MES_DIR / "Relatorio Contas a Pagar - Normalizado.xlsx"
    )

# O arquivo com prefixo "~$" e temporario do Excel (lock). Se ele for informado,
# convertemos para o nome real removendo o prefixo.
ARQUIVO_TEMP_EXCEL = ARQUIVO_NORMALIZADO_PRINCIPAL.with_name(f"~${ARQUIVO_NORMALIZADO_PRINCIPAL.name}")
ARQUIVO_RESUMO_FALLBACK = REFS_DIR / "outros" / "resumo_fatura_normalizado.xlsx"

# FECHAMENTO_ODBC_*.xlsx = gabarito de colunas ODBC (ordem e layout). O mes do arquivo nao importa.
FECHAMENTO_DIR = REFS_DIR / "Fechamento"
candidatos_modelo = sorted(FECHAMENTO_DIR.glob("FECHAMENTO_ODBC_*.xlsx"))
if not candidatos_modelo:
    candidatos_modelo = sorted(REFS_DIR.glob("FECHAMENTO_ODBC_*.xlsx"))
if not candidatos_modelo:
    candidatos_modelo = sorted((REFS_DIR / "outros").glob("FECHAMENTO_ODBC_*.xlsx"))
if not candidatos_modelo:
    raise FileNotFoundError(
        f"Nenhum FECHAMENTO_ODBC_*.xlsx em {FECHAMENTO_DIR.resolve()}, {REFS_DIR.resolve()} ou outros/"
    )
ARQUIVO_MODELO_FECHAMENTO = candidatos_modelo[-1]
print(f"Gabarito de colunas ODBC: {ARQUIVO_MODELO_FECHAMENTO}")

ARQUIVO_SAIDA_FECHAMENTO = REFS_DIR / "outros" / "resumo_fatura_fechamento_padrao.xlsx"
ARQUIVO_SAIDA_BRACOFER   = SUPPLY_MES_DIR / "BRACOFER - resumo_fatura_fechamento_padrao.xlsx"

def _norm_col(c: str) -> str:
    return str(c).strip().lower().replace(" ", "_")

def _to_float(v):
    if pd.isna(v):
        return pd.NA
    s = str(v).strip()
    if not s:
        return pd.NA
    s = s.replace("R$", "").replace(" ", "")
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return pd.NA


def carregar_fonte_segunda_normalizacao() -> tuple[pd.DataFrame, Path]:
    arquivo_real = ARQUIVO_NORMALIZADO_PRINCIPAL
    if not arquivo_real.exists() and ARQUIVO_TEMP_EXCEL.exists():
        arquivo_real = ARQUIVO_TEMP_EXCEL.with_name(ARQUIVO_TEMP_EXCEL.name.replace("~$", "", 1))

    if arquivo_real.exists():
        df = pd.read_excel(arquivo_real)
        df.columns = [_norm_col(c) for c in df.columns]
        return df, arquivo_real

    raw = pd.read_excel(ARQUIVO_RESUMO_FALLBACK)
    raw.columns = [_norm_col(c) for c in raw.columns]

    # Fallback: arquivo em formato chave-valor (nome, descricao)
    if set(raw.columns) >= {"nome", "descricao"} and len(raw.columns) == 2:
        kv = {
            str(k).strip().lower(): v
            for k, v in zip(raw["nome"], raw["descricao"])
            if pd.notna(k)
        }
        df = pd.DataFrame([
            {
                "numero_documento": kv.get("número", kv.get("numero", "")),
                "fornecedor": kv.get("nome", ""),
                "emissao": kv.get("emissão", kv.get("emissao", "")),
                "vencimento": kv.get("vencimento", ""),
                "pagamento": "",
                "valor_documento": _to_float(kv.get("valor")),
                "liquido": _to_float(kv.get("valor")),
                "empresa": "",
                "centro_custos_descricao": "",
                "classificacao_financeira_codigo": "",
                "classificacao_financeira_descricao": "",
                "observacao": "",
            }
        ])
        return df, ARQUIVO_RESUMO_FALLBACK

    return raw.copy(), ARQUIVO_RESUMO_FALLBACK

fonte_df, fonte_path = carregar_fonte_segunda_normalizacao()
_cols_modelo = pd.read_excel(ARQUIVO_MODELO_FECHAMENTO, nrows=0).columns.tolist()
# Modelo original: data_nf, data_pagamento. Inclui data_vecto (Vencimento) entre elas.
if "data_vecto" not in _cols_modelo:
    _k = _cols_modelo.index("data_nf") + 1
    modelo_cols = _cols_modelo[:_k] + ["data_vecto"] + _cols_modelo[_k:]
else:
    modelo_cols = _cols_modelo

print(f"Fonte carregada: {fonte_path.resolve()}")
print(f"Gabarito ODBC (colunas): {ARQUIVO_MODELO_FECHAMENTO.resolve()}")
print(f"Linhas na fonte: {len(fonte_df)}")
print("Colunas da fonte:", fonte_df.columns.tolist())

Gabarito de colunas ODBC: ..\..\02-Referencias\Fechamento\FECHAMENTO_ODBC_2026_05.xlsx
Fonte carregada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Supply_bracofer\Maio\Relatorio Contas a Pagar 05-2026 - Normalizado.xlsx
Gabarito ODBC (colunas): C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Fechamento\FECHAMENTO_ODBC_2026_05.xlsx
Linhas na fonte: 266
Colunas da fonte: ['numero_documento', 'descricao_documento', 'observacao', 'fornecedor', 'emissao', 'vencimento', 'pagamento', 'valor_documento', 'desconto', 'juros', 'multa', 'devolucao', 'liquido', 'empresa', 'classificacao_financeira_codigo', 'classificacao_financeira_descricao', 'classificacao_financeira_valor', 'centro_custos_descricao', 'centro_custos_valor']


## Conversao para layout FECHAMENTO_ODBC

Regras aplicadas nesta conversao:

- `empresa` -> `filial`
- `liquido` -> `valor_pago`
- `valor_documento` -> `valor_nf`
- Datas (relatorio bruto):
  - `emissao` -> `data_nf`
  - `vencimento` -> `data_vecto` (coluna extra apos o modelo ODBC)
  - `pagamento` -> `data_pagamento` (somente Pgto.; nao misturar com Vecto.)
- Centro de custo por descricao (ramo 1.3.1.x — Bracofer / Presidente Prudente):
  - `administrativo` -> `1.3.1.1`
  - `comercial`      -> `1.3.1.2`
  - `operacional`    -> `1.3.1.3`
  - `logistica`      -> `1.3.1.4`
  - `corte e dobra`  -> `1.3.1.5`
- Plano de Contas: mapeamento explicito Supply → SAGI (ver celula abaixo)

## Plano de Contas — Mapeamento Supply → SAGI

Os codigos de Classificacao Financeira do Supply **nao** seguem o padrao SAGI. Esta celula define o mapeamento explicito. Codigos nao encontrados no mapa serao marcados com prefixo `[NM]` no relatorio final para revisao manual.

Referencia de categorias e armadilhas da base consolidada: `Analise Base Financeira.md` (ex.: uso de `codcdc` / familias `7.5.x` para despesas administrativas). O codigo Supply **2010101** vem sempre como descricao generica **Servicos**; o mapeamento fixo para uma unica conta SAGI concentrava lancamentos indevidos (ver celula de codigo seguinte).

Cadastro oficial exportado do SAGI: `sagi_rel_plano_conta.csv` e `sagi_rel_centro_custo.csv` na pasta `02-Referencias/`. A celula seguinte le esses arquivos para **alinhar textos** de conta (e, na conversao do layout, descricoes de CC `1.3.1.x`) ao cadastro SAGI.

| Codigo Supply | Descricao Supply           | Cod SAGI  | Descricao SAGI                    |
|---------------|----------------------------|-----------|-----------------------------------|
| 301           | Mercadoria p/Revenda       | 6.5.1     | COMPRA P/ REVENDA                 |
| 309           | Mercadoria p/Industrializ. | 6.5.2     | INDUSTRIALIZACAO (cadastro SAGI)  |
| 20305         | Reembolso de Clientes      | 7.9.1     | DEVOLUCAO A CREDOR                |
| 2010101       | Servicos (veja codigo)     | (varios)  | Roteamento por fornecedor/textos  |
| 2010203       | (Supply)                   | 7.5.5     | MATERIAL DE ESCRITORIO            |
| 2020103       | Desp. e Materiais Diversos | 7.5.5     | MATERIAL DE ESCRITORIO            |
| 2020202       | Impostos, licencas e Taxas | 7.5.17    | TAXAS                             |
| 2020204       | Manut./reparos adm.        | 7.5.6     | MANUT. E REPAROS - ESTRUT. ADM.   |
| 2020205       | Despesas de Viagens        | 7.1.8     | DESPESAS DE VIAGEM                |
| 2040101       | Aluguel                    | 7.5.31    | ALUGUEL ADMINISTRATIVO            |
| 2040103       | Energia eletrica           | 7.5.2     | ENERGIA ELETRICA                  |
| 2040107       | Material escritorio        | 7.5.5     | MATERIAL DE ESCRITORIO            |
| 2040115       | Publicidade                | 7.2.5     | PUBLICIDADE E PROPAGANDA          |
| 2040119       | Brindes                    | 7.2.4     | BRINDES                           |
| 2040120       | Despesas                   | 7.11.1    | OPERACOES ENTRE EMPRESAS          |
| 2040201       | Juros pagos                | 7.5.23    | JUROS E MULTAS                    |
| 2040202       | Tarifas Bancarias          | 7.5.22    | DESPESAS BANCARIAS                |
| 2040208       | Taxa Adm Cartao            | 7.5.22    | DESPESAS BANCARIAS                |
| 2050101       | Salario                    | 7.3.1     | SALARIOS                          |
| 2050108       | Rescisoes                  | 7.3.21    | RESCISOES                         |
| 2050109       | Pensao Alimenticia         | 7.3.21    | RESCISOES                         |
| 2050201       | Assistencia Medica         | 7.3.6     | PLANO DE SAUDE                    |
| 2050202       | Transporte colaboradores | 7.3.10    | TRANSPORTE DE COLABORADORES       |
| 2050203       | Refeicoes                  | 7.3.5     | ALIMENTACAO DO TRABALHADOR        |
| 2050204       | Cesta Basica               | 7.3.8     | CESTA BASICA                      |
| 2050206       | Emprestimos                | 7.3.16    | EMPRESTIMO CONSIGNADO             |
| 2060201       | ICMS / tributos            | 7.4.12    | ICMS                              |
| 2060402       | FGTS                       | 7.3.2     | FGTS                              |
| 2060403       | Sindical                   | 7.3.13    | SINDICATOS                        |
| 2060405       | Seguro Trabalho            | 7.3.9     | SEGURANCA DO TRABALHO             |


In [29]:
from pathlib import Path
import re
import pandas as pd

_REFS_SAGI = globals().get("REFS_DIR")
if _REFS_SAGI is None:
    _REFS_SAGI = Path("../../02-Referencias").resolve()


def _carregar_sagi_plano_conta(path: Path) -> dict[str, str]:
    """Le sagi_rel_plano_conta.csv (export SAGI): codigo;descricao;..."""
    out: dict[str, str] = {}
    if not path.exists():
        return out
    raw = path.read_text(encoding="latin-1", errors="replace")
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith(";"):
            continue
        m = re.match(r"^(\d+(?:\.\d+)*);([^;]+)", line)
        if not m:
            continue
        cod, desc = m.group(1), m.group(2).strip()
        if desc and "R.SOCIAL" not in desc and "Rela" not in desc and "Código" not in desc:
            out[cod] = desc
    return out


def _carregar_sagi_centro_custo(path: Path) -> dict[str, str]:
    """Le sagi_rel_centro_custo.csv (export SAGI)."""
    out: dict[str, str] = {}
    if not path.exists():
        return out
    raw = path.read_text(encoding="latin-1", errors="replace")
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith(";"):
            continue
        if "R.SOCIAL" in line or "Rela" in line:
            continue
        raw_parts = line.split(";")
        cod = raw_parts[0].strip()
        desc = ""
        for p in raw_parts[1:]:
            if p.strip():
                desc = p.strip()
                break
        if not cod or not desc or not cod[0].isdigit():
            continue
        out[cod] = desc
    return out


_F_PLANO_SAGI = _REFS_SAGI / "sagi_rel_plano_conta.csv"
_F_CC_SAGI = _REFS_SAGI / "sagi_rel_centro_custo.csv"
_SAGI_PLANO_DESC = _carregar_sagi_plano_conta(_F_PLANO_SAGI)
_SAGI_CC_DESC = _carregar_sagi_centro_custo(_F_CC_SAGI)
print(
    f"Referencias SAGI: {len(_SAGI_PLANO_DESC)} contas ({_F_PLANO_SAGI.name}), "
    f"{len(_SAGI_CC_DESC)} centros ({_F_CC_SAGI.name})."
)


def _enriquecer_plano_sagi(pc: dict | None) -> dict | None:
    """Alinha a descricao ao cadastro do plano de contas SAGI (CSV), quando existir."""
    if not pc:
        return None
    cod = str(pc.get("cod", "")).strip()
    out = dict(pc)
    if cod and cod in _SAGI_PLANO_DESC:
        out["desc"] = _SAGI_PLANO_DESC[cod]
    return out


MAPA_PC_BRACOFER = {
    "301":     {"cod": "6.5.1",  "desc": "COMPRA P/ REVENDA"},
    "309":     {"cod": "6.5.2",  "desc": "INDUSTRIALIZACAO"},
    "20305":   {"cod": "7.9.1",  "desc": "DEVOLUCAO A CREDOR"},
    # 2010101 "Servicos" e generico no Supply — nao mapear aqui (ver _mapear_2010101_servicos).
    "2010203": {"cod": "7.5.5",  "desc": "MATERIAL DE ESCRITÓRIO"},
    "2020103": {"cod": "7.5.5",  "desc": "MATERIAL DE ESCRITORIO"},
    "2020202": {"cod": "7.5.17", "desc": "TAXAS"},
    "2020204": {"cod": "7.5.6",  "desc": "MANUTENÇÕES E REPAROS - ESTRUTURA ADMINISTRATIVA"},
    "2020205": {"cod": "7.1.8",  "desc": "DESPESAS DE VIAGEM"},
    "2040101": {"cod": "7.5.31", "desc": "ALUGUEL ADMINISTRATIVO"},
    "2040103": {"cod": "7.5.2",  "desc": "ENERGIA ELÉTRICA"},
    "2040107": {"cod": "7.5.5",  "desc": "MATERIAL DE ESCRITÓRIO"},
    "2040115": {"cod": "7.2.5",  "desc": "PUBLICIDADE E PROPAGANDA"},
    "2040119": {"cod": "7.2.4",  "desc": "BRINDES"},
    "2040120": {"cod": "7.11.1", "desc": "OPERACOES ENTRE EMPRESAS"},
    "2040201": {"cod": "7.5.23", "desc": "JUROS E MULTAS"},
    "2040202": {"cod": "7.5.22", "desc": "DESPESAS BANCARIAS"},
    "2040208": {"cod": "7.5.22", "desc": "DESPESAS BANCARIAS"},
    "2050101": {"cod": "7.3.1",  "desc": "SALARIOS"},
    "2050108": {"cod": "7.3.21", "desc": "RESCISOES"},
    "2050109": {"cod": "7.3.21", "desc": "RESCISOES"},
    "2050201": {"cod": "7.3.6",  "desc": "PLANO DE SAUDE"},
    "2050202": {"cod": "7.3.10", "desc": "TRANSPORTE DE COLABORADORES"},
    "2050203": {"cod": "7.3.5",  "desc": "ALIMENTACAO DO TRABALHADOR"},
    "2050204": {"cod": "7.3.8",  "desc": "CESTA BASICA"},
    "2050206": {"cod": "7.3.16", "desc": "EMPRESTIMO CONSIGNADO"},
    "2060201": {"cod": "7.4.12", "desc": "ICMS"},
    "2060402": {"cod": "7.3.2",  "desc": "FGTS"},
    "2060403": {"cod": "7.3.13", "desc": "SINDICATOS"},
    "2060405": {"cod": "7.3.9",  "desc": "SEGURANCA DO TRABALHO"},
}


def _norm_key_cod(cod_supply) -> str:
    if cod_supply is None or (isinstance(cod_supply, float) and pd.isna(cod_supply)):
        return ""
    s = str(cod_supply).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


def _mapear_2010101_servicos(contexto: str) -> dict:
    """Supply 2010101 = 'Servicos' (descricao generica). Antes tudo ia para 7.5.16 (alarme)."""
    import re

    t = re.sub(r"\s+", " ", (contexto or "").strip().lower())

    def hit(*palavras: str) -> bool:
        return any(p in t for p in palavras)

    if hit(
        "alarme",
        "cftv",
        "cctv",
        "circuito fechado",
        "videomonitor",
        "camera ip",
        "cameras ",
        "câmeras ",
        "sprinkler",
        "deteccao de incendio",
        "detecção de incendio",
        "racci",
    ):
        return {"cod": "7.5.16", "desc": "ALARME E MONITORAMENTO"}
    if hit("google", "microsoft 365", "office 365", "hospedagem de site", "linkedin ads", "meta ads", "facebook ads"):
        return {"cod": "7.5.20", "desc": "INTERNET"}
    if hit(
        "telecom",
        "telefonia",
        "fibra",
        "banda larga",
        "rede neutra",
        "telmex",
        "efix telecom",
        "jmn de comunic",
        "samm tecnologia",
    ):
        return {"cod": "7.5.3", "desc": "TELECOMUNICAÇÕES"}
    if hit("softland", "tidsoft", "ikatec", "software", "licenca de software", "licença de software", "sistemas especificos", "sistemas específicos"):
        return {"cod": "7.5.18", "desc": "SISTEMAS"}
    if hit("contabil", "contábil", "ossucci"):
        return {"cod": "7.5.7", "desc": "HONORÁRIOS CONTÁBEIS"}
    if hit("advocac", "advogad", "juridic", "jurídic"):
        return {"cod": "7.5.8", "desc": "HONORÁRIOS ADVOCATÍCIOS"}
    if hit("treinamento", "curso", "fundacao de ciencia", "fundação de ciência", "tecnologia e ensino", "consultor", "capacitacao", "capacitação"):
        return {"cod": "7.5.9", "desc": "CONSULTORIA"}
    if hit("cemig", "cpfl", "enel ", "energia eletrica", "energia elétrica", "conta de luz"):
        return {"cod": "7.5.2", "desc": "ENERGIA ELÉTRICA"}
    if hit("eletric", "eletrônic", "eletronic") and not hit("cemig", "cpfl", "enel ", "energia eletrica", "energia elétrica", "conta de luz"):
        return {"cod": "7.5.6", "desc": "MANUTENÇÕES E REPAROS - ESTRUTURA ADMINISTRATIVA"}
    if hit("embalagem", "embalagens"):
        return {"cod": "7.5.5", "desc": "MATERIAL DE ESCRITÓRIO"}
    if hit(
        "comercio de aliment",
        "comércio de aliment",
        "alimentos ltda",
        "cafe cruzeiro",
        "usina ",
        "lanches",
        "restaur",
        "refeicao",
        "refeição",
        "sanna",
    ):
        return {"cod": "7.5.29", "desc": "LANCHES"}
    if hit("clinica", "clínica", " medica", " médica", "laboratorio", "laboratório", "polivida"):
        return {"cod": "7.3.6", "desc": "PLANO DE SAÚDE"}
    if hit("limpeza", "faxina", "dedetiz", "higieniz"):
        return {"cod": "7.5.28", "desc": "SERVIÇO DE LIMPEZA DO ESCRITÓRIO"}
    if hit("segur", "vigilancia", "vigilância") and not hit("alarme", "cftv", "cctv"):
        return {"cod": "7.5.10", "desc": "SEGURANÇA E VIGILÂNCIA"}
    if hit("correio", "sedex"):
        return {"cod": "7.5.11", "desc": "CORREIOS"}
    if hit("grupo lidera", "associacao", "associação"):
        return {"cod": "7.5.26", "desc": "DESPESAS SOCIAIS E DOAÇÕES"}
    return {"cod": "7.5.9", "desc": "CONSULTORIA"}


def mapear_plano_contas_bracofer(cod_supply, contexto: str = "") -> dict | None:
    """Retorna {'cod': ..., 'desc': ...} no padrao SAGI, ou None se nao mapeado."""
    chave = _norm_key_cod(cod_supply)
    if chave == "2010101":
        pc = _mapear_2010101_servicos(contexto)
    else:
        pc = MAPA_PC_BRACOFER.get(chave)
    return _enriquecer_plano_sagi(pc)


print(f"MAPA_PC_BRACOFER: {len(MAPA_PC_BRACOFER)} entradas fixas + 2010101 (Servicos) + descricoes do CSV SAGI.")


Referencias SAGI: 0 contas (sagi_rel_plano_conta.csv), 0 centros (sagi_rel_centro_custo.csv).
MAPA_PC_BRACOFER: 29 entradas fixas + 2010101 (Servicos) + descricoes do CSV SAGI.


In [30]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import parse_valor_fechamento, parse_data_fechamento, gravar_fechamento_excel

def _pick(row, *keys, default=""):
    for k in keys:
        if k in row and pd.notna(row[k]) and str(row[k]).strip() != "":
            return row[k]
    return default

def _definir_cod_centro(descricao_origem: str) -> str:
    """
    BRACOFER / Presidente Prudente — 5 CCs analiticos (ramo 1.3.1.x):
      1.3.1.1 ADMINISTRATIVO
      1.3.1.2 COMERCIAL
      1.3.1.3 OPERACIONAL
      1.3.1.4 LOGISTICA
      1.3.1.5 CORTE E DOBRA
    """
    txt = str(descricao_origem or "").strip().lower()
    if "administrativo" in txt:
        return "1.3.1.1"
    if "comercial" in txt:
        return "1.3.1.2"
    if "operacional" in txt:
        return "1.3.1.3"
    if "logistica" in txt or "log\u00edstica" in txt:
        return "1.3.1.4"
    if "corte" in txt and "dobra" in txt:
        return "1.3.1.5"
    return "1.3.1.1"  # fallback para Administrativo

def _hierarquia_cc(cod_cc: str):
    partes = str(cod_cc).split(".")
    if len(partes) < 4:
        return cod_cc, cod_cc, cod_cc, cod_cc
    n1 = ".".join(partes[:2])
    n2 = ".".join(partes[:3])
    n3 = ".".join(partes[:4])
    n4 = cod_cc
    return n1, n2, n3, n4

_MAPA_DESC_CC = {
    "1.3.1.1": "ADMINISTRATIVO",
    "1.3.1.2": "COMERCIAL",
    "1.3.1.3": "OPERACIONAL",
    "1.3.1.4": "LOGISTICA",
    "1.3.1.5": "CORTE E DOBRA",
}

def _descricao_hierarquia(cod_cc: str):
    """
    Hierarquia validada para o ramo 1.3.1.x (Bracofer / Presidente Prudente).
    CCs analiticos: 1.3.1.1 a 1.3.1.5.
    Descricoes preferencialmente do arquivo sagi_rel_centro_custo.csv.
    """
    cc_key = str(cod_cc)
    fb = _MAPA_DESC_CC.get(cc_key, "ADMINISTRATIVO")
    _cc = globals().get("_SAGI_CC_DESC") or {}
    n3_desc = _cc.get(cc_key, fb)
    return {
        "segmento": "BRACOFER",
        "n1_desc": "BRACOFER",
        "n2_desc": "PRESIDENTE PRUDENTE",
        "n3_desc": n3_desc,
        "n4_desc": n3_desc,
    }

# Recarrega a fonte aqui para evitar problema de ordem de execucao do notebook.
fonte_df_exec, fonte_path_exec = carregar_fonte_segunda_normalizacao()
print(f"Fonte usada nesta execucao: {fonte_path_exec.resolve()}")
print(f"Linhas lidas: {len(fonte_df_exec)}")

if len(fonte_df_exec) == 0:
    raise ValueError(
        "A fonte da segunda normalizacao veio vazia. "
        "Execute novamente as celulas anteriores e confirme se o arquivo "
        "'Relatorio Contas a Pagar <periodo> - Normalizado.xlsx' existe e tem linhas."
    )

pcs_nao_mapeados = []
linhas_saida = []
for i, row in fonte_df_exec.iterrows():
    row = {k: row[k] for k in fonte_df_exec.columns}

    filial = _pick(row, "filial", "empresa", default="G3S")
    titulo = _pick(row, "titulo", "numero_documento", "numero", default=f"DOC-{i+1}")

    # Protecao extra: ignora eventual rodape herdado da fonte.
    titulo_txt = str(titulo).strip().lower()
    if "desenvolvido por softland" in titulo_txt or "softland sistemas" in titulo_txt:
        continue

    # colunas equivalentes solicitadas:
    # empresa=filial, liquido=valor_pago, valor_documento=valor_nf
    valor_nf = _to_float(_pick(row, "valor_nf", "valor_documento", "valor", default=pd.NA))
    valor_pago = _to_float(_pick(row, "valor_pago", "liquido", "valor", default=valor_nf))

    centro_desc = _pick(row, "centro_custos_descricao", "centro_origem", "centro_custo", "departamento", default="")
    cod_cc = _definir_cod_centro(centro_desc)
    n1, n2, n3, n4 = _hierarquia_cc(cod_cc)

    nova = {c: pd.NA for c in modelo_cols}
    nova["id"] = i + 1
    nova["filial"] = filial
    nova["titulo"] = str(titulo)
    nova["valor_nf"] = valor_nf
    nova["valor_pago"] = -abs(valor_pago) if valor_pago is not None and not pd.isna(valor_pago) else pd.NA
    vc = valor_nf if pd.isna(valor_pago) else valor_pago
    nova["valor_conta"] = -abs(vc) if vc is not None and not pd.isna(vc) else pd.NA

    nova["n1_cod_centro_custo"] = n1
    nova["n2_cod_centro_custo"] = n2
    nova["n3_cod_centro_custo"] = n3
    nova["n4_cod_centro_custo"] = n4

    desc_h = _descricao_hierarquia(cod_cc)
    nova["Segmento"] = desc_h["segmento"]
    nova["n1_centro_custo"] = desc_h["n1_desc"]
    nova["n2_centro_custo"] = desc_h["n2_desc"]
    nova["n3_centro_custo"] = desc_h["n3_desc"]
    nova["n4_centro_custo"] = desc_h["n4_desc"]

    nova["n1_CC"] = f"{n1} {desc_h['n1_desc']}"
    nova["n2_CC"] = f"{n2} {desc_h['n2_desc']}"
    nova["n3_CC"] = f"{n3} {desc_h['n3_desc']}"
    nova["n4_CC"] = f"{n4} {desc_h['n4_desc']}"

    nova["data_nf"] = parse_data_fechamento(_pick(row, "data_nf", "emissao", "emissão", default=""))
    nova["data_vecto"] = _pick(row, "data_vecto", "vencimento", default="")
    # Pgto. real: usar somente pagamento (antes vencimento vinha antes por ordem do _pick).
    nova["data_pagamento"] = parse_data_fechamento(_pick(row, "data_pagamento", "pagamento", default=""))
    nova["credor_forn_cli_func"] = _pick(row, "credor_forn_cli_func", "fornecedor", "nome", default="")
    nova["observacao"] = _pick(row, "observacao", default="")

    cod_supply_raw = _pick(row, "classificacao_financeira_codigo", "cod_conta", default="")
    ctx_pc = " ".join(
        str(_pick(row, k, default="")).strip()
        for k in (
            "classificacao_financeira_descricao",
            "fornecedor",
            "descricao_documento",
            "observacao",
        )
    )
    pc = mapear_plano_contas_bracofer(cod_supply_raw, ctx_pc)
    if pc:
        nova["cod_conta"] = pc["cod"]
        nova["conta"]     = pc["desc"]
    else:
        nova["cod_conta"] = cod_supply_raw
        nova["conta"]     = _pick(row, "classificacao_financeira_descricao", "conta", default="")
        if cod_supply_raw:
            pcs_nao_mapeados.append({
                "linha": i + 1,
                "cod_supply": cod_supply_raw,
                "desc_supply": _pick(row, "classificacao_financeira_descricao", "conta", default=""),
                "fornecedor": _pick(row, "fornecedor", "credor_forn_cli_func", default=""),
            })
    if str(nova["cod_conta"]).strip() and str(nova["conta"]).strip():
        nova["cod_conta-descr"] = f"{nova['cod_conta']} {nova['conta']}"

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=modelo_cols)

periodo_saida = globals().get("SUFIXO_PERIODO", "MM-AAAA")
_supply_out = (
    globals().get("SUPPLY_MES_DIR")
    or globals().get("SUPPLY_DIR")
    or Path("../../02-Referencias/Supply_bracofer")
)
ARQUIVO_SAIDA_FINAL = _supply_out / f"BRACOFER_fechamento_{periodo_saida}.xlsx"

for destino in [ARQUIVO_SAIDA_FECHAMENTO, ARQUIVO_SAIDA_BRACOFER, ARQUIVO_SAIDA_FINAL]:
    destino.parent.mkdir(parents=True, exist_ok=True)
    try:
        gravar_fechamento_excel(fechamento_df, destino, sheet_name="fechamento_normalizado")
        print(f"Arquivo gerado: {destino.resolve()}")
    except PermissionError:
        print(f"[AVISO] Arquivo em uso, pulando: {destino.name}")

print(f"\nLinhas geradas: {len(fechamento_df)}")
fechamento_df.head(10)


Fonte usada nesta execucao: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Supply_bracofer\Maio\Relatorio Contas a Pagar 05-2026 - Normalizado.xlsx
Linhas lidas: 266
Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\outros\resumo_fatura_fechamento_padrao.xlsx
Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Supply_bracofer\Maio\BRACOFER - resumo_fatura_fechamento_padrao.xlsx
Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\Supply_bracofer\Maio\BRACOFER_fechamento_05-2026.xlsx

Linhas geradas: 266


,id,Segmento,n1_cod_centro_custo,n1_centro_custo,n1_CC,n2_cod_centro_custo,n2_centro_custo,n2_CC,n3_cod_centro_custo,n3_centro_custo,n3_CC,n4_cod_centro_custo,n4_centro_custo,n4_CC,cod_conta,conta,cod_conta-descr,filial,titulo,valor_nf,valor_pago,valor_conta,observacao,data_nf,data_vecto,data_pagamento,cod_credor_forn_cli_func,credor_forn_cli_func,Origem,Sistema,Dados auxiliares,Valor Oficial,DE-PARA1,DE-PARA2,CUSTEIO VARIÁVEL
0,1,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,7.5.20,INTERNET,7.5.20 INTERNET,3,000000000-001,58.69,-58.69,-58.69,REFERENTE A NF 000000000 | [GIULIANA] EMBRATEL...,2026-05-22,25/05/26,2026-05-25,<NA>,02667694000140 - TELMEX DO BRASIL S/A,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,7.5.9,CONSULTORIA,7.5.9 CONSULTORIA,3,000000003-001,391.96,-391.96,-391.96,REFERENTE A NF 000000003 | PEDIDO(S) ASSOCIADO...,2026-04-05,15/05/26,2026-05-15,<NA>,46007927000154 - ALEX FERREIRA ARCE 33602634809,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,3,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.2,COMERCIAL,1.3.1.2 COMERCIAL,1.3.1.2,COMERCIAL,1.3.1.2 COMERCIAL,6.5.1,COMPRA P/ REVENDA,6.5.1 COMPRA P/ REVENDA,3,000000005-001,1500.00,-1500.00,-1500.00,REFERENTE A NF 000000005 | PEDIDO(S) ASSOCIADO...,2026-05-18,25/05/26,2026-05-25,<NA>,66235712000106 - INDUSTRIA DE ROLDANAS SEMENSI...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,4,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,7.5.9,CONSULTORIA,7.5.9 CONSULTORIA,3,000000039-001,700.00,-700.00,-700.00,REFERENTE A NF 000000039 | PEDIDO(S) ASSOCIADO...,2026-05-05,15/05/26,2026-05-15,<NA>,57360768000193 - PATRICIA ALVES TOLEDO,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,5,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.3,OPERACIONAL,1.3.1.3 OPERACIONAL,1.3.1.3,OPERACIONAL,1.3.1.3 OPERACIONAL,7.5.9,CONSULTORIA,7.5.9 CONSULTORIA,3,000000042-001,4000.00,-4000.00,-4000.00,REFERENTE A NF 000000042 | PEDIDO(S) ASSOCIADO...,2026-05-05,15/05/26,2026-05-15,<NA>,55599387000136 - SANTANA E DA SILVA CONSULTORI...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,6,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,7.5.20,INTERNET,7.5.20 INTERNET,3,000000107-001,1000.00,-1000.00,-1000.00,REFERENTE A NF 000000107 | PEDIDO(S) ASSOCIADO...,2026-05-05,15/05/26,2026-05-15,<NA>,52596265000106 - GALAS TRAFAGO PAGO LTDA,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,7,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,7.2.5,PUBLICIDADE E PROPAGANDA,7.2.5 PUBLICIDADE E PROPAGANDA,3,000000119-001,1000.00,-1000.00,-1000.00,REFERENTE A NF 000000119 | PEDIDO(S) ASSOCIADO...,2026-12-05,15/05/26,2026-05-15,<NA>,30905135000167 - MARAIZA FEITOSA MARTINS PARDI...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,8,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,7.5.9,CONSULTORIA,7.5.9 CONSULTORIA,3,000000196-001,11880.00,-11880.00,-11880.00,REFERENTE A NF 000000196 | PEDIDO(S) ASSOCIADO...,2026-05-05,15/05/26,2026-05-18,<NA>,00395519000116 - FUNDACAO DE CIENCIA TECNOLOGI...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,9,BRACOFER,1.3,BRACOFER,1.3 BRACOFER,1.3.1,PRESIDENTE PRUDENTE,1.3.1 PRESIDENTE PRUDENTE,1.3.1.1,ADMINISTRATIVO,1.3.1.1 ADMINISTRATIVO,1.3.1.1,ADMINISTRATIV

## Relatorio de Planos de Contas nao mapeados

Lista os codigos do Supply que nao foram encontrados no `MAPA_PC_BRACOFER`. Esses itens foram gravados no arquivo de saida com o codigo Supply original (prefixo `[NM]` ausente, mas rastreavel pela lista abaixo). Adicione as entradas faltantes ao mapa na celula anterior e re-execute.

In [31]:
if not pcs_nao_mapeados:
    print("[OK] Todos os Planos de Contas foram mapeados para o padrao SAGI. Nenhuma pendencia.")
else:
    df_pendentes = pd.DataFrame(pcs_nao_mapeados)
    resumo = df_pendentes.groupby(["cod_supply", "desc_supply"]).size().reset_index(name="ocorrencias")
    resumo = resumo.sort_values("ocorrencias", ascending=False)

    print(f"[PENDENTE] {len(resumo)} codigo(s) Supply sem mapeamento SAGI ({len(pcs_nao_mapeados)} linha(s) afetadas):\n")
    print(resumo.to_string(index=False))
    print()
    print("Para corrigir: adicione as entradas no dicionario MAPA_PC_BRACOFER (celula 9) e re-execute.")
    print()
    print("Detalhe por linha:")
    display(df_pendentes)

[PENDENTE] 6 codigo(s) Supply sem mapeamento SAGI (7 linha(s) afetadas):

 cod_supply                   desc_supply  ocorrencias
    2040122                     Uniformes            2
      20301            Fretes de Entregas            1
      40102            Projetos Materiais            1
    2020206             Despesas Diversas            1
    2040106               Telefonia Movel            1
    2040109 Material de Higiene / Limpeza            1

Para corrigir: adicione as entradas no dicionario MAPA_PC_BRACOFER (celula 9) e re-execute.

Detalhe por linha:


,linha,cod_supply,desc_supply,fornecedor
0,14,2040109,Material de Higiene / Limpeza,63034197000108 - HONNESTA DISTRIBUIDORA LTDA
1,44,2040106,Telefonia Movel,51582889000101 - EFIX TELECOM TELECOMUNICAÇOES...
2,74,40102,Projetos Materiais,38311793000132 - LIFE FORMA INCORPORADA LTDA
3,75,20301,Fretes de Entregas,00428307001240 - EXPRESSO SAO MIGUEL
4,125,2020206,Despesas Diversas,60855160000225 - ELETROREDE MATERIAIS ELETRICO...
5,128,2040122,Uniformes,08148733000180 - CAMISETAS.COM CONFECCOES PRUD...
6,129,2040122,Uniformes,08148733000180 - CAMISETAS.COM CONFECCOES PRUD...
